In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")

DB_DIR = "./db/cafe_db"
MENU_TXT = "./data/cafe_menu.txt"

print("DB_DIR:", DB_DIR)
print("MENU_TXT exists?:", os.path.exists(MENU_TXT))

DB_DIR: ./db/cafe_db
MENU_TXT exists?: True


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.docstore.document import Document
import os, pathlib

def ensure_vector_db(db_dir=DB_DIR, menu_txt_path=MENU_TXT):
    """
    - db_dir에 FAISS 인덱스가 있으면 로드
    - 없으면 menu_txt를 읽어 간단히 인덱스 생성 후 저장 (4-1이 없을 때 대비용)
    """
    embeddings = OpenAIEmbeddings()

    if os.path.exists(db_dir):
        try:
            vs = FAISS.load_local(db_dir, embeddings, allow_dangerous_deserialization=True)
            print(f"✅ Loaded Vector DB from: {db_dir}")
            return vs
        except Exception as e:
            print("Vector DB load failed, rebuilding...", e)

   
    print("⚠️ Vector DB not found. Building a minimal index from cafe_menu.txt ...")
    assert os.path.exists(menu_txt_path), f"'{menu_txt_path}' 파일이 필요합니다."

    raw = pathlib.Path(menu_txt_path).read_text(encoding="utf-8")
    chunks = [c.strip() for c in raw.split("\n\n") if c.strip()]

    docs = []
    for chunk in chunks:
        if "메뉴:" in chunk:
            name_line = next((ln for ln in chunk.splitlines() if ln.startswith("메뉴:")), "메뉴: Unknown")
            menu_name = name_line.split("메뉴:")[-1].strip()
            docs.append(Document(page_content=chunk, metadata={"menu_name": menu_name}))

    assert len(docs) > 0, "menu_txt에서 '메뉴:'를 포함한 항목을 하나 이상 만들어주세요."

    vs = FAISS.from_documents(docs, embeddings)
    os.makedirs(db_dir, exist_ok=True)
    vs.save_local(db_dir)
    print(f"✅ Built & Saved Vector DB → {db_dir} (docs={len(docs)})")
    return vs

menu_db = ensure_vector_db()


c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\mylangchain-app-SBe-Yh6W-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded Vector DB from: ./db/cafe_db


In [3]:
from typing import List, Dict, Optional, Literal, Any
from typing_extensions import TypedDict
from langgraph.graph.message import MessagesState 
from langchain_core.documents import Document

class CafeState(MessagesState, TypedDict, total=False):
    
    intent: Optional[Literal["메뉴문의","가격문의","추천요청","기타"]]
    retrieved: Optional[List[Document]] 
    extracted: Optional[Dict[str, Any]]  
    user_query: Optional[str]            


In [4]:
import re
from langchain_core.documents import Document as LCDocument

def semantic_search(query: str, k: int = 4) -> List[LCDocument]:
    """Vector DB에서 의미론적 검색"""
    return menu_db.similarity_search(query, k=k)

def extract_menu_info(doc: LCDocument) -> dict:
    """
    Vector DB 문서에서 구조화된 정보 추출
    기대 포맷(예시):
    메뉴: 아메리카노
    가격: ₩4,000
    설명: 고소한...
    옵션: ICE/HOT
    """
    content = doc.page_content
    menu_name = doc.metadata.get("menu_name", "Unknown")

    price_match = re.search(r"(₩[\d,]+)", content)
    desc_match  = re.search(r"설명:\s*(.+?)(?:\n|$)", content, re.DOTALL)
    option_match= re.search(r"(옵션|옵션:|선택):\s*(.+?)(?:\n|$)", content)

    return {
        "name": menu_name,
        "price": price_match.group(1) if price_match else "가격 정보 없음",
        "description": desc_match.group(1).strip() if desc_match else "설명 없음",
        "options": option_match.group(2).strip() if option_match else None,
        "raw": content,
    }


In [ ]:
def classify_intent(text: str) -> Literal["메뉴문의","가격문의","추천요청","기타"]:
    t = text.strip().lower()
    
    price_kw = ["가격", "얼마", "비싸", "저렴", "원", "비용"]
    rec_kw   = ["추천", "추천해", "뭐가 좋아", "인기", "베스트", "시그니처"]
    menu_kw  = ["메뉴", "성분", "설명", "칼로리", "옵션", "핫", "아이스", "디카페인", "사이즈"]

    if any(k in t for k in price_kw):
        return "가격문의"
    if any(k in t for k in rec_kw):
        return "추천요청"
    if any(k in t for k in menu_kw):
        return "메뉴문의"
    return "기타"


In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)


In [7]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END

def node_classify(state: CafeState) -> CafeState:
    last = state["messages"][-1].content
    intent = classify_intent(last)
    state["intent"] = intent
    state["user_query"] = last
    return state

def node_retrieve(state: CafeState) -> CafeState:
    intent = state.get("intent") or "기타"
    user_message = state.get("user_query", "")

    
    docs: List[LCDocument] = []

    if intent == "가격문의":
        
        docs = semantic_search("메뉴 가격", k=5)
        
        docs2 = semantic_search(user_message, k=2)
        docs = (docs or []) + (docs2 or [])

    elif intent == "추천요청":
        docs = semantic_search(user_message, k=3)
        if not docs:
            docs = semantic_search("인기 메뉴", k=3)

    elif intent == "메뉴문의":
        docs = semantic_search(user_message, k=4)

    else:  
        docs = semantic_search(user_message, k=3)

    state["retrieved"] = docs[:5] if docs else []
    return state


def node_extract(state: CafeState) -> CafeState:
    docs = state.get("retrieved") or []
    extracted_list = []
    for d in docs[:2]:
        extracted_list.append(extract_menu_info(d))
    state["extracted"] = {"items": extracted_list} if extracted_list else {}
    return state


def node_respond(state: CafeState) -> CafeState:
    intent = state.get("intent", "기타")
    user_q = state.get("user_query", "")
    docs = state.get("retrieved") or []
    ext = state.get("extracted") or {}

    
    def compact_item(it):
        base = f"- {it['name']} / {it.get('price','가격 정보 없음')}"
        if it.get("description") and it["description"] != "설명 없음":
            base += f" — {it['description']}"
        if it.get("options"):
            base += f" (옵션: {it['options']})"
        return base

    items_text = ""
    if ext.get("items"):
        items_text = "\n".join(compact_item(it) for it in ext["items"])

    sys = SystemMessage(content=(
        "너는 카페 점원 AI야. 정중하고 간결하게 답하고, 한국어로 답해. "
        "가능하면 검색된 메뉴 명/가격/설명을 활용해 정확히 안내해. "
        "가격은 '₩#,###' 형식을 유지해."
    ))

    
    if intent == "가격문의":
        user = HumanMessage(content=(
            f"질문: {user_q}\n\n아래는 검색된 가격 관련 정보야.\n"
            f"{items_text if items_text else '검색 결과가 거의 없어요.'}\n\n"
            "요청: 사용자가 물은 가격을 중심으로 핵심만 요약해줘. "
            "여러 메뉴가 있으면 2~3개만 대표로 알려줘."
        ))
    elif intent == "추천요청":
        user = HumanMessage(content=(
            f"질문: {user_q}\n\n아래는 추천에 참고할 수 있는 검색 정보야.\n"
            f"{items_text if items_text else '추천 관련 직접 정보가 적어요.'}\n\n"
            "요청: 2~3개 메뉴를 상황에 맞게 추천하고, 짧게 이유를 덧붙여줘."
        ))
    elif intent == "메뉴문의":
        user = HumanMessage(content=(
            f"질문: {user_q}\n\n아래는 해당 메뉴 설명/옵션 등이야.\n"
            f"{items_text if items_text else '해당 메뉴에 대한 직접 정보가 적어요.'}\n\n"
            "요청: 사용자 질문에 맞게 메뉴 특징/옵션/가격 위주로 답해줘."
        ))
    else:
        user = HumanMessage(content=(
            f"질문: {user_q}\n\n검색 요약:\n"
            f"{items_text if items_text else '직접 관련 결과가 부족합니다.'}\n\n"
            "요청: 가능한 한 도움되는 정보를 간단히 정리해줘. "
            "필요하면 추가 질문도 제안해."
        ))

    ai = llm.invoke([sys, user])
    
    messages = state.get("messages", [])
    messages.append(AIMessage(content=ai.content))
    state["messages"] = messages
    return state


In [9]:
from langgraph.graph import StateGraph

graph = StateGraph(CafeState)

graph.add_node("classify", node_classify)
graph.add_node("retrieve", node_retrieve)
graph.add_node("extract", node_extract)
graph.add_node("respond", node_respond)

graph.set_entry_point("classify")

graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "extract")
graph.add_edge("extract", "respond")
graph.add_edge("respond", END)

app = graph.compile()


In [10]:
from langchain_core.messages import HumanMessage, AIMessage

def run_turn(app, history: List = None, user_text: str = "") -> List:
    """
    - history: 이전까지의 messages (Human/AIMessage 리스트)
    - user_text: 이번 사용자 입력
    반환: 최신 messages
    """
    history = history or []
    history = history + [HumanMessage(content=user_text)]

    state: CafeState = {"messages": history}
    out = app.invoke(state)

    return out["messages"]

messages = []


In [11]:
messages = run_turn(app, messages, "라떼 가격 어떻게 돼?")
print(messages[-1].content)

카페라떼의 가격 정보는 확인되지 않았습니다. 부드러운 우유 풍미와 에스프레소의 조화가 특징입니다. 모카 또한 가격 정보는 없지만, 초콜릿의 달콤함과 커피의 조화가 매력적입니다. 추가적인 정보가 필요하시면 말씀해 주세요!


In [12]:
messages = run_turn(app, messages, "오늘같이 더운 날 아이스 추천 좀!")
print(messages[-1].content)

더운 날에 추천드리는 아이스 음료는 다음과 같습니다:

1. **아이스 그린티라떼** - 진한 녹차 풍미와 부드러운 우유가 어우러져 시원하게 즐기기 좋은 음료입니다.

2. **아이스 초코라떼** - 달콤하고 진한 초코 맛으로, 더위를 잊게 해줄 시원한 선택입니다.

이 두 가지 음료로 더위를 날려보세요!


In [13]:
messages = run_turn(app, messages, "콜드브루에 디카페인 옵션 있어?")
print(messages[-1].content)

죄송하지만, 현재 콜드브루에는 디카페인 옵션이 제공되지 않습니다. 대신 디카페인 아메리카노를 추천드립니다. 디카페인 아메리카노는 카페인 부담 없이 아메리카노의 맛을 그대로 즐길 수 있는 메뉴입니다. 추가적인 문의가 있으시면 언제든지 말씀해 주세요!


In [14]:
messages = run_turn(app, messages, "공부하기 조용한 좋은 좌석이 어디 쪽이야?")
print(messages[-1].content)

조용한 공부하기 좋은 좌석은 카페의 구석자리나 창가 쪽을 추천드립니다. 일반적으로 사람들이 덜 지나다니는 곳이 더 조용합니다. 

또한, 음료로는 카페라떼와 콜드브루가 있습니다. 카페라떼는 부드러운 우유 풍미와 에스프레소의 조화가 특징이고, 콜드브루는 장시간 저온 추출로 산미가 낮고 깔끔한 여운이 있습니다. 

더 궁금한 점이 있으시면 언제든지 말씀해 주세요!
